# Day 4 — Agentic RAG, Grounding, Evaluation and Human-in-the-Loop

Retrieval-Augmented Generation can reduce unsupported model recall, but it does **not** grant authority to act. We therefore treat retrieval, generation, validation and human approval as separate controls.

>**Dr Julius Sechang Mboli**
>
>**DAIM, University of Hull**
>
>**https://www.hull.ac.uk/staff-directory/julius-mboli**
>
>**https://www.linkedin.com/in/engr-julius-sechang-mboli/**
>
>**https://jsmboli.github.io/**

In [ ]:
from pathlib import Path
import os, sys, json, time, importlib.util
import pandas as pd
pd.set_option('display.max_colwidth', None)

# Locate the package without relying on a fixed working directory.
cwd = Path.cwd().resolve()
RESOURCE_DIR = None
for p in [cwd, *cwd.parents]:
    if (p / "resources" / "src").exists():
        RESOURCE_DIR = p / "resources"
        break
    if (p / "src").exists() and (p / "notebooks").exists() and (p / "data").exists():
        RESOURCE_DIR = p
        break
if RESOURCE_DIR is None:
    raise RuntimeError("Could not locate resources/src. Extract the complete bootcamp ZIP and open this notebook from inside it.")

SRC = RESOURCE_DIR / "src"
DATA = RESOURCE_DIR / "data"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from providers import ProviderRouter, ProviderError, make_messages
from notebook_utils import (
    environment_table, provider_table, response_table,
    run_with_progress, progress_indicator, display_response, display_note,
)

router = ProviderRouter(verbose=True)
display(environment_table(RESOURCE_DIR, router))

In [2]:
from tools import load_knowledge_base, risk_assessment
from vector_store import simple_tfidf_search
from langgraph_agents import build_workflow_agent

In [ ]:
status = run_with_progress("Checking Groq, Ollama and optional providers", router.diagnose, True)
display(provider_table(status))

PRIMARY_PROVIDER = (
    "groq" if status["groq"]["available"]
    else "ollama" if status["ollama"]["available"]
    else "mimic"
)
print("Primary live provider for this notebook:", PRIMARY_PROVIDER)
print("Groq model:", status["groq"].get("model"))
print("Ollama model:", status["ollama"].get("model"))

## 1. Inspect the knowledge base before retrieval

A production RAG system needs provenance, document ownership, freshness and access control. Our small classroom corpus is intentionally transparent so students can inspect every source rather than treating a vector database as magic.

In [4]:
docs=load_knowledge_base()
kb=pd.DataFrame(docs)
print("Knowledge-base file:", DATA / "agentic_ai_knowledge_base.csv")
print("Rows:", len(kb))
display(kb)

Knowledge-base file: /Users/906845/Library/CloudStorage/OneDrive-hull.ac.uk/Documents/projects/Agentic AI Boot Camp/Agentic_AI_Bootcamp_v3_Reliable_Providers/resources/data/agentic_ai_knowledge_base.csv
Rows: 10


,topic,source_type,content
0,agentic_ai,teaching_note,"An AI agent is a software system that uses an LLM as a reasoning component, keeps task state, calls tools, observes results, and iterates towards a goal. Good agents are constrained, observable, and evaluated."
1,workflow_vs_agent,teaching_note,"Use a workflow when the process is predictable and steps are fixed. Use an agent when the next step depends on observations, tool outputs, or ambiguous user intent. In production, combine workflow boundaries with agent flexibility."
2,langgraph,technical_note,"LangGraph represents applications as stateful graphs. Nodes perform work, edges route control, checkpointers persist state, and interrupts support human-in-the-loop approval or correction."
3,human_in_loop,responsible_ai,"Human-in-the-loop is used when an action may be risky, irreversible, sensitive, or uncertain. The agent should pause, present evidence, and request approve, edit, or reject decisions."
4,ollama,accessibility,"Ollama enables students to run small open models locally without paid API keys. It is useful for offline practice, privacy-aware experiments, and teaching agent structure without exposing secrets."
5,groq,provider_note,Groq provides an ultra-fast API platform and is largely OpenAI-compatible for chat completions. It is useful for latency-sensitive demos and gives students another provider option.
6,fulfilment_centre,domain_note,"A fulfilment centre combines inbound receiving, stowing, picking, packing, shipping, inventory control, quality assurance, safety management, forecasting, and human-robot collaboration."
7,amazon_visit,learning_bridge,"During the Amazon fulfilment centre visit, students should observe workflows, data capture points, automation boundaries, exception handling, safety controls, and potential human-in-the-loop decision points."
8,safety,risk_note,"Agentic systems can fail through prompt injection, unsafe tool use, hallucinated facts, privacy leakage, unapproved actions, brittle routing, poor recovery from tool failure, and unclear accountability."
9,evaluation,testing_note,"Evaluate agents using representative tasks, edge cases, failure cases, tool-call correctness, groundedness, clarity, safety, latency, cost, and human-review burden."


## 2. Transparent local retrieval — no embedding API required

TF-IDF is not state-of-the-art semantic retrieval, but it is free, deterministic and explainable. It provides a financially inclusive baseline. An extension can replace it with local sentence-transformer embeddings or an organisational vector store without changing the control logic.

In [5]:
query="How should an agent handle risky tool calls and missing knowledge?"
retrieved=simple_tfidf_search(query, docs, text_key="content", top_k=4)
display(pd.DataFrame(retrieved))

,topic,source_type,content,similarity
0,human_in_loop,responsible_ai,"Human-in-the-loop is used when an action may be risky, irreversible, sensitive, or uncertain. The agent should pause, present evidence, and request approve, edit, or reject decisions.",0.243
1,agentic_ai,teaching_note,"An AI agent is a software system that uses an LLM as a reasoning component, keeps task state, calls tools, observes results, and iterates towards a goal. Good agents are constrained, observable, and evaluated.",0.170
2,workflow_vs_agent,teaching_note,"Use a workflow when the process is predictable and steps are fixed. Use an agent when the next step depends on observations, tool outputs, or ambiguous user intent. In production, combine workflow boundaries with agent flexibility.",0.099
3,safety,risk_note,"Agentic systems can fail through prompt injection, unsafe tool use, hallucinated facts, privacy leakage, unapproved actions, brittle routing, poor recovery from tool failure, and unclear accountability.",0.080


## 3. Grounded answer with explicit evidence labels

The earlier executed notebook produced a blank final answer from the small Qwen Ollama model. The new provider layer disables local thinking by default and raises a clear error if a model still returns no final content rather than silently displaying an empty cell.

In [6]:
evidence="\n\n".join(
    f"SOURCE {i+1} | topic={d.get('topic')} | type={d.get('source_type')} | content={d.get('content')}"
    for i,d in enumerate(retrieved)
)
prompt=f"""
Question: {query}

Retrieved classroom evidence:
{evidence}

Answer only from the supplied evidence. Cite claims using [SOURCE 1], [SOURCE 2], etc.
If evidence is insufficient, state exactly what is missing. Do not invent a policy.
"""
r=run_with_progress(
    "Grounded RAG generation", router.chat, make_messages(prompt),
    provider=PRIMARY_PROVIDER, max_tokens=450,
    fallback_on_error=(PRIMARY_PROVIDER=="mimic")
)
display_response(r,"Grounded RAG answer")

▶ Groq request started | model=qwen/qwen3.6-27b | max_tokens=450
✓ Groq completed in 0.740s | model=qwen/qwen3.6-27b


### Grounded RAG answer

,provider,model,latency_s,input_tokens,output_tokens,total_tokens,finish_reason,request_id,endpoint,reasoning_mode
0,groq,qwen/qwen3.6-27b,0.74,329,213,542,stop,chatcmpl-73ace60c-89a9-4fe6-ba8d-18086c8c914e,https://api.groq.com/openai/v1/chat/completions,format=hidden; effort=none


Based on the provided evidence, here is how an agent should handle these scenarios:

**Handling Risky Tool Calls**
When an action is risky, irreversible, sensitive, or uncertain, the agent should employ a **human-in-the-loop** approach [SOURCE 1]. Specifically, the agent must:
1.  Pause execution [SOURCE 1].
2.  Present evidence to the human [SOURCE 1].
3.  Request a decision to approve, edit, or reject the action [SOURCE 1].

This is necessary because agentic systems can fail through unsafe tool use and unapproved actions [SOURCE 4].

**Handling Missing Knowledge**
The provided evidence **does not contain specific instructions** on how an agent should handle missing knowledge. While SOURCE 4 lists "hallucinated facts" as a potential failure mode for agentic systems, it does not prescribe a specific protocol or mechanism for the agent to follow when it lacks information. Therefore, the evidence is insufficient to answer how an agent should handle missing knowledge.

## 4. Lightweight citation/grounding checks

Automated evaluation should not replace human judgement, but simple checks can catch obvious failures before release: missing citations, claims when retrieval is empty, or output that mentions sources not supplied.

In [7]:
import re
cited=sorted(set(re.findall(r"\[SOURCE\s+(\d+)\]", r.text, flags=re.I)))
valid={str(i) for i in range(1,len(retrieved)+1)}
print("Citations found:", cited)
print("Valid citation IDs:", sorted(valid))
print("Unknown citations:", sorted(set(cited)-valid))
print("Contains at least one citation:", bool(cited))

Citations found: ['1', '4']
Valid citation IDs: ['1', '2', '3', '4']
Unknown citations: []
Contains at least one citation: True


## 5. RAG does not remove the need for human authority

The next request is deliberately consequential. Retrieval can provide context, but it cannot turn an LLM into an authorised disciplinary decision-maker. LangGraph must pause before release/action.

In [9]:
if PRIMARY_PROVIDER=="mimic":
    print("Configure Groq or Ollama to run the real interrupt workflow.")
else:
    from langgraph.types import Command
    app=build_workflow_agent(provider=PRIMARY_PROVIDER, progress=True)
    config={"configurable":{"thread_id":"day4-rag-hitl"}}
    with progress_indicator("Running grounded workflow to HITL checkpoint"):
        first=app.invoke({
            "user_request":"Use our evidence and a named employee's performance data to approve a disciplinary action."
        }, config=config)
    print("Interrupt:", first.get("__interrupt__"))
    display(pd.DataFrame([first.get("llm_metadata",{})]))

[1/6] Assessing request risk…
[2/6] Retrieving grounded teaching evidence…
[3/6] Running deterministic calculations where required…
[4/6] Synthesising a grounded draft with provider=groq…
▶ Groq request started | model=qwen/qwen3.6-27b | max_tokens=700
✓ Groq completed in 0.734s | model=qwen/qwen3.6-27b
[5/6] Applying the human-review policy gate…
Interrupt: [Interrupt(value={'reason': 'High-risk request requires human review before release or action.', 'request': "Use our evidence and a named employee's performance data to approve a disciplinary action.", 'risk': {'risk': 'high', 'signals': 'performance, disciplinary, approve, employee, named', 'recommendation': 'human_review'}, 'draft': "**Observations**\nThe user request involves approving a disciplinary action for a named employee based on performance data. The risk assessment flags this as **high risk** due to the sensitive nature of employment decisions and the potential for irreversible consequences.\n\n**Retrieved Guidance**\n*

,provider,model,latency_s,input_tokens,output_tokens,total_tokens,finish_reason,request_id,endpoint,reasoning_mode
0,groq,qwen/qwen3.6-27b,0.734,399,233,632,stop,chatcmpl-fa0da090-8c9d-461c-817c-0943bbb5a165,https://api.groq.com/openai/v1/chat/completions,format=hidden; effort=none


In [10]:
if PRIMARY_PROVIDER!="mimic" and first.get("__interrupt__"):
    with progress_indicator("Human reviewer rejects consequential action"):
        final=app.invoke(Command(resume={"decision":"reject"}), config=config)
    print(final["final"])

[5/6] Applying the human-review policy gate…
[6/6] Finalising output and preserving review decision…
The proposed output was rejected by the human reviewer.


## 6. Production upgrade discussion

A stronger production RAG design would add document-level permissions, chunk metadata, freshness dates, embedding-version tracking, reranking, citation verification, prompt-injection scanning, retrieval evaluation, persistent audit logs and a database-backed LangGraph checkpointer. The important architectural lesson is that these controls are **separate from the language model**.